## **<mark>Import the required packages and functions</mark>**

In [7]:
from pyspark.sql.functions import to_date, year, month, current_date, quarter, hour,\
date_format, weekofyear, avg, max, min, count, col, round, when, dayofmonth, monotonically_increasing_id, lag,lpad,concat
from pyspark.sql.window import Window
from delta.tables import DeltaTable
from datetime import datetime
import uuid
from pyspark.sql.types import *

StatementMeta(, f347f32a-109b-43b5-94af-4c4ccb8e445c, 11, Finished, Available, Finished, False)

## **<mark>Audit variable creation</mark>**

In [8]:
run_id = str(uuid.uuid4())
pipeline_run_id = ""      # Pipeline parameter
notebook_name = "Gold Notebook"
start_time = datetime.now()
status = "none"
records_processed = 0
error_message = "none"

StatementMeta(, f347f32a-109b-43b5-94af-4c4ccb8e445c, 12, Finished, Available, Finished, False)

## **<mark>gold layer ETL process inside notebook audit</mark>**

In [9]:
try:

    #=============================
    # Incremental load logic for gold layer
    #=============================


    # here we take only the new rows for incremental load using the "silver layer max ingection time processed" 

    spark.sql("CREATE SCHEMA IF NOT EXISTS gold")

    if spark.catalog.tableExists("LH_Weather_Update.gold.fact_weather"):
    
        gold_last_processed_time = (spark.sql("""SELECT MAX(ingestion_time) AS max_time
            FROM LH_Weather_Update.gold.fact_weather""").first()["max_time"])

        df = spark.sql(f"""SELECT *FROM LH_Weather_Update.silver.weather_selected
            WHERE ingestion_time > TIMESTAMP('{gold_last_processed_time}')""")
    
    else:
        df = spark.sql("""SELECT *FROM LH_Weather_Update.silver.weather_selected""")

    #=========================================
    # creating temerature column in Fahrenheit
    #=========================================

    df = df.withColumnRenamed("Temperature", "Temperature_C") \
            .withColumn("Temperature_F", round((col("Temperature_C") * 9/5) + 32, 2))

    #=========================================
    # creating Date Dimension table
    #=========================================
    dim_date = df.select(to_date("Datetime_IST").alias("Date")).distinct()\
        .withColumn("Date_key", monotonically_increasing_id())\
            .withColumn("Year", year("Date"))\
            .withColumn("Quarter", quarter("Date"))\
            .withColumn("Month", month("Date"))\
            .withColumn("MonthName", date_format("Date","MMMM"))\
            .withColumn("WeekOfYear", weekofyear("Date"))\
            .withColumn("Day", dayofmonth("Date"))\
            .withColumn("DayName", date_format("Date","EEEE"))

#-------------------------------------------------------------------------------
# this is mandatery step in this incremental process without this process a you will get a new rows dupicates on every time run
# this code helpes to find a new rows already in a existing table or not, if a new rows already in a existing table,
# then its not going to append that row, it helps to avoid duplication of data.

    if not spark.catalog.tableExists("gold.dim_date"):
        dim_date.write.format("delta").mode("overwrite").option("mergeSchema","true").saveAsTable("gold.dim_date")
    else:

        existing_dates = spark.table("gold.dim_date")

        new_dim_date = (dim_date.alias("n").join(existing_dates.select("Date").alias("e")
        ,on="Date",how="left_anti"))

        new_dim_date.write.format("delta").mode("append").option("mergeSchema","true").saveAsTable("gold.dim_date")
    #===============================
    # Creating Time Dimension Table
    #===============================
    if not spark.catalog.tableExists("gold.dim_time"):

        dim_time = (df.select(hour("Datetime_IST").alias("Hour_24")).distinct()
                .withColumn("Hour_12",when(col("Hour_24") == 0, 12)
                    .when(col("Hour_24") > 12, col("Hour_24") - 12)
                    .otherwise(col("Hour_24")))
                .withColumn("AM_PM", when(col("Hour_24") < 12, "AM")
                    .otherwise("PM"))
                .withColumn("TimeLabel",concat(
                    lpad(when(col("Hour_24") == 0, 12)
                    .when(col("Hour_24") > 12, col("Hour_24") - 12)
                    .otherwise(col("Hour_24"))
                    .cast("string"),2,"0"),
                when(col("Hour_24") < 12, " AM")
                .otherwise(" PM")))
                .withColumn("TimeBand",
                    when(col("Hour_24").between(5, 11), "Morning")
                    .when(col("Hour_24").between(12, 16), "Afternoon")
                    .when(col("Hour_24").between(17, 20), "Evening")
                    .otherwise("Night"))
                .withColumn("Time_key",monotonically_increasing_id())
                .select("Time_key",
                "Hour_24",
                "Hour_12",
                "AM_PM",
                "TimeLabel",
                "TimeBand"))

        dim_time.write.format("delta").mode("overwrite").saveAsTable("gold.dim_time")
    else:
        dim_time = spark.table("gold.dim_time")

    # gold_dim_time already have max rows, because we have a 00:00 to 23:00 hours we got that rows in a One day data,
    #so we no need secound day data

    #==================================
    # Creating Location Dimension Table
    #==================================

    dim_location = (df.select("latitude","longitude","city","state","country","timezone").distinct()
            .withColumn("Location_key", monotonically_increasing_id()))

    if not spark.catalog.tableExists("gold.dim_location"):

        dim_location.write.format("delta").mode("overwrite").option("mergeSchema","true").saveAsTable("gold.dim_location")
    else:
        existing_location = spark.table("gold.dim_location")

        new_dim_location = (dim_location.alias("n").join(existing_location.select("latitude","longitude").alias("e")
        ,on=["latitude","longitude"],how="left_anti"))

        new_dim_location.write.format("delta").mode("append").option("mergeSchema","true").saveAsTable("gold.dim_location")
    
    #=============================
    # Creating Daily summary table
    #=============================

    # Daily summary
    daily_summary = df.groupBy(to_date(col("Datetime_IST")).alias("date")) \
            .agg(round(avg(col("Temperature_C")), 2).alias("avg_temp_c"),
            round(avg(col("Temperature_f")), 2).alias("avg_temp_f"),
            max(col("Temperature_C")).alias("max_temp"),
            min(col("Temperature_C")).alias("min_temp"),
            count("*").alias("count_records"))

    daily_summary = daily_summary.withColumn("previous_avg_temp_c",lag("avg_temp_c").over(Window.orderBy("date")))
    daily_summary = daily_summary.withColumn("temp_change",round(col("avg_temp_c") - col("previous_avg_temp_c"), 2))

    # Heat Wave Analysis
    daily_summary = daily_summary.withColumn("weather_condition",
            when(col("avg_temp_c") >= 40,"Extreme Heat")
            .when(col("avg_temp_c") >= 35,"Heat Wave")
            .when(col("avg_temp_c") >= 30,"Warm")
            .otherwise("Normal"))

    # Temparature band
    daily_summary = daily_summary.withColumn("temp_band",
            when(col("avg_temp_c") < 20, "<20")
            .when(col("avg_temp_c") < 25, "20-25")
            .when(col("avg_temp_c") < 30, "25-30")
            .when(col("avg_temp_c") < 35, "30-35")
            .otherwise("35+"))

    #========================================================================================================================

    # always use if condition its helps to handal first run

    if not spark.catalog.tableExists("gold.daily_summary"):
        daily_summary.write.format("delta").saveAsTable("gold.daily_summary")
    else:
        daily_delta = DeltaTable.forName(spark,"gold.daily_summary")
        (daily_delta.alias("t")
            .merge(daily_summary.alias("s"),
            "t.date = s.date")
            .whenMatchedUpdateAll()
            .whenNotMatchedInsertAll()
            .execute())

    # here i am using a merge option insted of using join because i want to update the existing rows also its not possible in join
    # its support UPSET option (update + insert)

    #=============================
    # creating Monthly summary table
    #=============================
    
    # Monthly summary

    month_summary = df.groupBy(year(col("Datetime_IST")).alias("year"),
            month(col("Datetime_IST")).alias("month")) \
            .agg(round(avg(col("Temperature_C")), 2).alias("avg_temp_c"),
            round(avg(col("Temperature_f")), 2).alias("avg_temp_f"),
            max(col("Temperature_C")).alias("max_temp_c"),
            min(col("Temperature_C")).alias("min_temp_c"))

    month_summary = month_summary.withColumn("previous_avg_temp_c",lag("avg_temp_c").over(Window.partitionBy("year").orderBy("month")))
    month_summary = month_summary.withColumn("temp_change",round(col("avg_temp_c") - col("previous_avg_temp_c"), 2))

#========================================================================================================================

    # always use if condition its helps to handal first run

    if not spark.catalog.tableExists("gold.monthly_summary"):
        month_summary.write.format("delta").mode("overwrite").saveAsTable("gold.monthly_summary")
    else:
        monthly_delta = DeltaTable.forName(spark,"gold.monthly_summary")
        (monthly_delta.alias("t")
            .merge(month_summary.alias("s"),
            "t.year = s.year AND t.month = s.month")
                .whenMatchedUpdateAll()
                .whenNotMatchedInsertAll()
                .execute())

# here i am using a merge option insted of using join because i want to update the existing rows also its not possible in join
# its support UPSET option (update + insert)

#=============================================================================================================================

    #=============================
    # creating Monthly summary table
    #=============================

    # Hourly analysis
    hourly_summary = df.groupBy(hour(col("Datetime_IST")).alias("hour_24")) \
            .agg(round(avg(col("Temperature_C")), 2).alias("avg_temp_c"),
            round(avg(col("Temperature_f")), 2).alias("avg_temp_f"),
            max(col("Temperature_C")).alias("max_temp_c"),
            min(col("Temperature_C")).alias("min_temp_c"))

    hourly_summary = hourly_summary.withColumn("previous_avg_temp_c",lag("avg_temp_c").over(Window.orderBy("hour_24")))
    hourly_summary = hourly_summary.withColumn("temp_change",round(col("avg_temp_c") - col("previous_avg_temp_c"), 2))

#========================================================================================================================

    # always use if condition its helps to handal first run

    if not spark.catalog.tableExists("gold.hourly_summary"):
        hourly_summary.write.format("delta").mode("overwrite").saveAsTable("gold.hourly_summary")
    else:
        hourly_delta = DeltaTable.forName(spark,"gold.hourly_summary")
        (hourly_delta.alias("t")
                .merge(hourly_summary.alias("s"),
                "t.hour_24 = s.hour_24")
                .whenMatchedUpdateAll()
                .whenNotMatchedInsertAll()
                .execute())

    # here i am using a merge option insted of using join because i want to update the existing rows also its not possible in join
    # its support UPSET option (update + insert)

    #=============================
    # Creating Fact weather table
    #=============================


    # selecting the requried column for fact table

    df = df.withColumn("Date", to_date("Datetime_IST"))\
        .withColumn("Hour_24", hour("Datetime_IST"))

    # join the date dimenstion table

    df = df.join(dim_date.select("Date_key","Date"),on="Date",how="left")

    # join the time dimention table

    df = df.join(dim_time.select("Time_key", "Hour_24"),on="Hour_24",how="left")

    #  join the location dimension table

    df = df.join(dim_location.select("Location_key","latitude","longitude"),on=["latitude", "longitude"],how="left")

    # selecting and organizinng the final columns for fact table
    fact_df = df.select(
        "Date_key",
        "Time_key",
        "Location_key",
        "Temperature_C",
        "Temperature_F",
        "generationtime_ms",
        "elevation",
        "ingestion_time")

    fact_df.write.format("delta").mode("append").option("mergeSchema","true").saveAsTable("gold.fact_weather")

    records_processed = df.count()

    status = "Success"

except Exception as e:

    status = "Failed"
    error_message = str(e)

    raise

finally:

    end_time = datetime.now()

    duration = (end_time - start_time).total_seconds()

    notebook_audit_df = spark.createDataFrame(
        [(
            str(uuid.uuid4()),
            pipeline_run_id,
            notebook_name,
            status,
            records_processed,
            start_time,
            end_time,
            duration,
            error_message,
            datetime.now()
        )],
        [
            "RunID",
            "PipelineRunID",
            "NotebookName",
            "Status",
            "RecordsProcessed",
            "StartTime",
            "EndTime",
            "DurationSeconds",
            "ErrorMessage",
            "CreatedOn"
        ]
    )

    notebook_audit_df.write.mode("append").saveAsTable("audit.notebook_audit")

    #================================================
    # Audit variables for Incremental load Audit table
    #================================================

    start_time = datetime.now()
    run_id = str(uuid.uuid4())
    pipeline_run_id = ""
    table_name = "gold.fact_weather"
    status = "Success"
    error_message = ""
    rows_inserted = 0
    rows_updated = 0

    # initialize variables
    previous_watermark = None
    current_watermark = None

    try:

    # previous watermark

        if spark.catalog.tableExists("gold.fact_weather"):

            previous_watermark = (
            spark.sql("""
            SELECT MAX(ingestion_time)
            FROM gold.fact_weather
            """).first()[0]
        )

    # rows processed in this run

        if fact_df is not None:
            rows_inserted = fact_df.count()
        else:
            rows_inserted = 0
            status = "Failed"

    # new watermark

        if rows_inserted > 0:

            current_watermark = (
                fact_df.agg({"ingestion_time":"max"})
                .first()[0])

        else:

            current_watermark = previous_watermark

            status = "Success"

    except Exception as e:

        status = "Failed"

        error_message = str(e)

        raise

    finally:

        end_time = datetime.now()

        duration = (end_time-start_time).total_seconds()

        audit_schema = StructType([
            StructField("RunID", StringType(), True),
            StructField("PipelineRunID", StringType(), True),
            StructField("TableName", StringType(), True),
            StructField("PreviousWatermark", TimestampType(), True),
            StructField("CurrentWatermark", TimestampType(), True),
            StructField("RowsInserted", IntegerType(), True),
            StructField("RowsUpdated", IntegerType(), True),
            StructField("Status", StringType(), True),
            StructField("StartTime", TimestampType(), True),
            StructField("EndTime", TimestampType(), True),
            StructField("DurationSeconds", DoubleType(), True),
            StructField("ErrorMessage", StringType(), True),
            StructField("CreatedOn", TimestampType(), True)])
        
        audit_df = spark.createDataFrame(
        [(run_id,
        pipeline_run_id,
        table_name,
        previous_watermark,
        current_watermark,
        rows_inserted,
        rows_updated,
        status,
        start_time,
        end_time,
        duration,
        error_message,
        datetime.now())],
        schema=audit_schema)

        audit_df.write.mode("append").option("mergeSchema", "true").saveAsTable("audit.incremental_load_audit")

StatementMeta(, f347f32a-109b-43b5-94af-4c4ccb8e445c, 13, Finished, Available, Finished, False)